# Monetary Policy Transmission Lags in Canada

**Research Question:** When the Bank of Canada unexpectedly hikes the overnight rate by **+100 bps (+1 percentage point)**, how many quarters does it take for the shock to peak in the **housing market (NHPI)** versus **headline inflation (CPI)** and **real output (GDP)**, and how much of that macroeconomic volatility is actually driven by monetary policy shocks?

This notebook walks through:
1. Historical Canadian macroeconomic series (`1992Q1`–`2026Q2`, inflation-targeting era)
2. Stationarity tests (ADF & KPSS) on log-levels vs. QoQ annualized growth rates
3. Lag selection criteria (AIC, BIC, HQIC, FPE) and SVAR residual diagnostics
4. Cholesky-identified **5-variable open-economy SVAR** Impulse Response Functions (IRFs) with 68% and 90% residual bootstrap bands
5. **Pre-COVID robustness check (`1992Q1`–`2019Q4`)** isolating normal monetary transmission from pandemic/supply-chain shocks
6. Forecast Error Variance Decomposition (FEVD) across 1- to 5-year horizons

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
PROCESSED = ROOT / "data" / "processed"
RESULTS = PROCESSED / "results"

levels = pd.read_csv(PROCESSED / "quarterly_levels.csv", index_col="quarter")
stationary = pd.read_csv(PROCESSED / "quarterly_stationary.csv", index_col="quarter")
stationarity_df = pd.read_csv(PROCESSED / "stationarity_tests.csv")

print(f"Quarterly levels shape: {levels.shape} ({levels.index.min()} to {levels.index.max()})")
levels.tail()

## 1. Historical Macroeconomic Overview (1992Q1–2026Q2)

In [ ]:
q_dates = pd.PeriodIndex(levels.index, freq="Q").to_timestamp()
s_dates = pd.PeriodIndex(stationary.index, freq="Q").to_timestamp()

fig, axes = plt.subplots(3, 2, figsize=(13, 9), sharex=True)
axes = axes.flatten()

series_specs = [
    (q_dates, levels["overnight_rate"], "Bank of Canada Overnight Rate (%)", "#1e3a8a"),
    (s_dates, stationary["cpi_yoy"], "Headline CPI Inflation (YoY %)", "#2563eb"),
    (s_dates, stationary["nhpi_yoy"], "New Housing Price Index — NHPI (YoY %)", "#dc2626"),
    (s_dates, stationary["gdp_yoy"], "Real GDP Growth (YoY %)", "#059669"),
    (q_dates, levels["usdcad"], "USD/CAD Exchange Rate (CAD per USD)", "#7c3aed"),
]

for i, (dt, ser, title, color) in enumerate(series_specs):
    ax = axes[i]
    ax.plot(dt, ser, color=color, lw=1.8)
    ax.axvspan(pd.Timestamp("2020-01-01"), pd.Timestamp("2022-01-01"), color="gray", alpha=0.14, label="COVID-19 shock")
    if "YoY" in title:
        ax.axhline(0, color="black", lw=0.8, ls="--")
    if "CPI" in title:
        ax.axhline(2.0, color="#dc2626", lw=0.9, ls=":", label="BoC 2% target")
    ax.set_title(title, fontweight="bold")
    ax.grid(alpha=0.25)
    if i in (0, 1):
        ax.legend(fontsize=8)

axes[5].axis("off")
fig.suptitle("Canadian Macroeconomic Panel — Inflation-Targeting Era", fontsize=13, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## 2. Stationarity Diagnostics (ADF & KPSS)

- **Levels:** `log_gdp`, `log_cpi`, `log_nhpi`, and `overnight_rate` are non-stationary I(1). Following Sims, Stock & Watson (1990) and Christiano, Eichenbaum & Evans (1999), we estimate our primary monetary SVAR in **detrended log-levels (`trend="ct"`)** so OLS remains consistent and preserves cointegrating dynamics while directly reporting cumulative `%` level impacts.
- **Transformed:** First-differenced QoQ annualized growth rates and `d_overnight_rate` are cleanly stationary I(0), providing a strict-stationarity cross-check.

In [ ]:
stationarity_df[stationarity_df["sample"].str.startswith("Full")][
    ["sample", "variable", "adf_stat", "adf_pvalue", "kpss_stat", "kpss_pvalue", "verdict"]
]

## 3. Lag Selection & SVAR Diagnostics

In [ ]:
models = ["full_levels", "precovid_levels", "full_stationary", "precovid_stationary"]
diag_rows = []
for m in models:
    meta = json.loads((RESULTS / m / "summary.json").read_text(encoding="utf-8"))
    d = meta["diagnostics"]
    diag_rows.append({
        "model": m,
        "sample": f"{meta['start_quarter']}–{meta['end_quarter']}",
        "T": meta["n_obs"],
        "lag_p": meta["selected_lag"],
        "trend": d["trend"],
        "stable": d["is_stable"],
        "max_eig": d["max_companion_eigenvalue"],
        "whiteness_p": d["whiteness_pvalue"],
        "shock_1sd_bps": round(meta["shock_1sd_pp"] * 100, 1),
    })
pd.DataFrame(diag_rows)

In [ ]:
print("Lag Selection — Full Sample Log-Levels:")
display(pd.read_csv(RESULTS / "full_levels" / "lag_selection.csv"))
print("\nLag Selection — Pre-COVID Log-Levels:")
display(pd.read_csv(RESULTS / "precovid_levels" / "lag_selection.csv"))

## 4. Impulse Response Functions (IRFs): Housing vs. Headline Inflation & Output

Cholesky recursive ordering (slow-moving to fast-moving):  
`Real GDP → Headline CPI → Housing (NHPI) → Overnight Rate → USD/CAD FX`

All responses are normalized to a **+100 bps (+1.0 percentage point)** unexpected Bank of Canada overnight rate hike.

In [ ]:
def load_irfs(model_name: str) -> dict[str, pd.DataFrame]:
    base = RESULTS / model_name
    return {
        "point": pd.read_csv(base / "irf_100bp.csv", index_col="horizon_q"),
        "lo68": pd.read_csv(base / "irf_lower_68.csv", index_col="horizon_q"),
        "hi68": pd.read_csv(base / "irf_upper_68.csv", index_col="horizon_q"),
        "lo90": pd.read_csv(base / "irf_lower_90.csv", index_col="horizon_q"),
        "hi90": pd.read_csv(base / "irf_upper_90.csv", index_col="horizon_q"),
        "fevd": pd.read_csv(base / "fevd_policy.csv", index_col="horizon_q"),
    }

full_lvl = load_irfs("full_levels")
pre_lvl = load_irfs("precovid_levels")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
targets = [
    ("log_gdp", "Real GDP (% level)"),
    ("log_cpi", "Headline CPI (% level)"),
    ("log_nhpi", "Housing Prices — NHPI (% level)"),
]
h = full_lvl["point"].index.values

for ax, (col, title) in zip(axes, targets):
    ax.axhline(0, color="black", lw=0.8, ls="--")
    ax.plot(h, full_lvl["point"][col], color="#1e3a8a", lw=2.2, label="Full (1992–2026)")
    ax.fill_between(h, full_lvl["lo68"][col], full_lvl["hi68"][col], color="#1e3a8a", alpha=0.14)
    ax.plot(h, pre_lvl["point"][col], color="#dc2626", lw=2.2, ls="--", label="Pre-COVID (1992–2019)")
    ax.fill_between(h, pre_lvl["lo68"][col], pre_lvl["hi68"][col], color="#dc2626", alpha=0.14)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Quarters after +100 bps hike")
    ax.set_xticks(range(0, 21, 4))
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8.5)

axes[0].set_ylabel("% deviation from baseline")
fig.suptitle("Full-Sample vs. Pre-COVID Monetary Transmission (68% Bootstrap Bands)", fontsize=13, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

## 5. Peak Transmission Lags & Forecast Error Variance Decomposition (FEVD)

In [ ]:
peaks = pd.read_csv(RESULTS / "all_models_peak_summary.csv")
peaks[[
    "model", "variable", "trough_quarter", "trough_response_per_100bp",
    "sig_68_negative", "fevd_q4_pct", "fevd_q8_pct", "fevd_q12_pct", "fevd_q20_pct"
]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
horizons = [4, 8, 12, 20]
x = np.arange(len(horizons))
w = 0.25

for ax, bundle, title in [
    (axes[0], full_lvl["fevd"], "Full Sample (1992–2026)"),
    (axes[1], pre_lvl["fevd"], "Pre-COVID Robustness (1992–2019)"),
]:
    ax.bar(x - w, [bundle.loc[q, "log_gdp"] for q in horizons], width=w, color="#059669", label="Real GDP")
    ax.bar(x, [bundle.loc[q, "log_nhpi"] for q in horizons], width=w, color="#dc2626", label="Housing (NHPI)")
    ax.bar(x + w, [bundle.loc[q, "log_cpi"] for q in horizons], width=w, color="#1d4ed8", label="Headline CPI")
    ax.set_title(f"FEVD Share Driven by Policy Shocks\n{title}", fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([f"Q{q}" for q in horizons])
    ax.set_xlabel("Forecast Horizon")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()

axes[0].set_ylabel("Variance Explained by Overnight Rate Shocks (%)")
fig.tight_layout()
plt.show()

## 6. Key Takeaways

1. **No Price Puzzle in Open-Economy Specification:** Including the USD/CAD exchange rate eliminates the classic price puzzle—headline CPI falls following an unexpected +100 bps BoC hike, reaching its trough around **Q9–Q11** (`-0.24%` pre-COVID, statistically significant at 68%; `-0.13%` full sample).
2. **Output vs. Housing vs. Inflation Lags & Magnitude:**
   - **Real GDP** contracts significantly, peaking at **Q9 (`-0.45%`)** pre-COVID and **Q11 (`-0.63%`)** in the full sample.
   - **New Housing Prices (NHPI)** exhibit a much larger cumulative decline—nearly **4× deeper than CPI** (`-0.95%` pre-COVID; `-0.47%` full sample)—with a drawn-out trough around **Q19–Q20**, reflecting contract-pricing inertia in newly built homes (while growth-rate deceleration hits earlier at **Q4–Q7** in the stationary VAR).
3. **How Much Volatility Is Driven by Monetary Policy (FEVD)?**
   - Over the normal pre-COVID inflation-targeting era (`1992–2019`), unexpected policy rate shocks explain **8.5% of Real GDP**, **9.9% of Headline CPI**, and **5.7% of NHPI** forecast error variance at the 5-year horizon (and **15.2%** of USD/CAD).
   - Including the COVID-19 and 2022–2023 supply-shock era (`1992–2026`) lowers monetary policy's FEVD share for CPI (`0.5%`) and NHPI (`0.7%`), underscoring that exogenous global supply and pandemic shocks dominated post-2020 price volatility.